# Create Table with Information about Locations for Queries on NewsAPI

This Jupyter Notebook creates a table with information about location of interest. 
In particular, location names in English and Arabic as well as URLs of Wikipedia articles about the locations.
These informations can be used to query all articles of interest on NewsAPI (https://newsapi.ai/). 

Note that there are different lists of District names for some of the countries included in this project. 
This table was created based on the UN OCHA shapefiles and before receiving the World Bank shapefile.

Author: Dominik Wielath (dominik.wielath@gmail.com)

DIME Artificial Intelligence (DIME AI) - World Bank Group

Date: 2024-09-10

In [ ]:
import pandas as pd
import geopandas as gpd
from eventregistry import *

In [ ]:
shapefiles_path = "../../data/shapefiles/"
data_path_newsapi = "../../data/newsapi/"

In [ ]:
mashreq_countries = ["Iraq", "Jordan", "Lebanon", "Palestine", "Syria"]

In [ ]:
# Setting the API key for Event Registry
News_API_key_dominik = '7ce3fa46-3bab-4460-a552-e3046d565492'
News_API_key_philipp = '9200e557-062b-411a-8b58-0592c0fac183'
News_API_key_dominik_wb = '34b2eba6-04d7-4ad0-9aad-06a4bb50fae6'
News_API_key = News_API_key_dominik_wb

er = EventRegistry(apiKey = News_API_key)

Adding admin level 0

In [ ]:
# Creating the dataframe with the concept and location URIs
mashreq_countries_df = pd.DataFrame(mashreq_countries, columns = ["country"])
mashreq_countries_df["uri"] = [er.getLocationUri(country) for country in mashreq_countries_df["country"].values]

mashreq_countries_df.loc[mashreq_countries_df["country"] == "Palestine", "uri"] = "https://en.wikipedia.org/wiki/State_of_Palestine"

mashreq_countries_df["location_name_ara"] = ""
mashreq_countries_df.loc[mashreq_countries_df["country"] == "Iraq","location_name_ara"] = "العراق"
mashreq_countries_df.loc[mashreq_countries_df["country"] == "Jordan","location_name_ara"] = "الأردن"
mashreq_countries_df.loc[mashreq_countries_df["country"] == "Lebanon","location_name_ara"] = "لبنان"
mashreq_countries_df.loc[mashreq_countries_df["country"] == "Palestine","location_name_ara"] = "فلسطين"
mashreq_countries_df.loc[mashreq_countries_df["country"] == "Syria","location_name_ara"] = "سوريا"

mashreq_countries_df.rename(columns = {"country": "location_name"}, inplace = True)
mashreq_countries_df["country"] = mashreq_countries_df["location_name"]
mashreq_countries_df["adm"] = 0

In [ ]:
palestine_df = mashreq_countries_df.iloc[[3]].copy().reset_index(drop = True)
palestine_df = pd.concat([palestine_df, palestine_df, palestine_df, palestine_df, palestine_df]).reset_index(drop = True)

palestine_df.loc[palestine_df.index == 1, "uri"] = "https://en.wikipedia.org/wiki/Palestine"
palestine_df.loc[palestine_df.index == 2, "uri"] = "https://en.wikipedia.org/wiki/Palestine_(region)"
palestine_df.loc[palestine_df.index == 3, "uri"] = "https://en.wikipedia.org/wiki/Palestinian_territories"
palestine_df.loc[palestine_df.index == 4, "uri"] = "https://en.wikipedia.org/wiki/Palestinian_Authority"

In [ ]:
mashreq_countries_df = mashreq_countries_df.loc[mashreq_countries_df["location_name"] != "Palestine"].reset_index(drop = True)
mashreq_countries_df = pd.concat([mashreq_countries_df, palestine_df]).reset_index(drop = True)

Adding admin level 1

In [ ]:
iraq_districts_shp_name = shapefiles_path + "administrative-boundaries/iraq/irq_admbnda_adm2_cso_20190603.shp"
jordan_districts_shp_name = shapefiles_path + "administrative-boundaries/jordan/jordan_clean.shp"
lebanon_districts_shp_name = shapefiles_path + "administrative-boundaries/lebanon/lbn_admbnda_adm2_cdr_20200810.shp"
palestine_districts_shp_name = shapefiles_path + "administrative-boundaries/palestine/palestine_clean.shp"
syria_districts_shp_name = shapefiles_path + "administrative-boundaries/syria/syr_admin2.shp"

In [ ]:
iraq_districts_shp = gpd.read_file(iraq_districts_shp_name)
jordan_districts_shp = gpd.read_file(jordan_districts_shp_name)
lebanon_districts_shp = gpd.read_file(lebanon_districts_shp_name)
palestine_districts_shp = gpd.read_file(palestine_districts_shp_name)
syria_districts_shp = gpd.read_file(syria_districts_shp_name)

In [ ]:
# English location names
iraq_provinces_shp_names = iraq_districts_shp[["ADM1_EN_cl", "ADM1_AR"]].drop_duplicates().reset_index(drop=True)
lebanon_provinces_shp_names = lebanon_districts_shp[["admin1Na_1", "ADM1_EN_cl"]].drop_duplicates().reset_index(drop=True)
syria_provinces_shp_names = syria_districts_shp[["ADM1_AR", "ADM1_EN_cl"]].drop_duplicates().reset_index(drop=True)
jordan_provinces = ["irbid", "ajloun", "jerash", "mafraq", "balqa", "amman", "zarqa", "madaba", "karak", "tafilah", "ma'an", "aqaba"]
jordan_provinces_ara = ["إربد", "عجلون", "جرش", "المفرق", "البلقاء", "عمان", "الزرقاء", "مادبا", "الكرك", "الطفيلة", "معان", "العقبة"]
palestine_provinces = palestine_districts_shp["ADM1_EN_cl"].unique()

In [ ]:
# Iraq
iraq_provinces_df = pd.DataFrame(data={"location_name": iraq_provinces_shp_names["ADM1_EN_cl"].values, "uri" : [er.getLocationUri(country) for country in iraq_provinces_shp_names["ADM1_EN_cl"].values],"location_name_ara":iraq_provinces_shp_names["ADM1_AR"].values, "adm" : 1, "country":"Iraq"})

iraq_provinces_df.loc[iraq_provinces_df["location_name"] == "baghdad", "uri"] = "https://en.wikipedia.org/wiki/Baghdad_Governorate"
iraq_provinces_df.loc[iraq_provinces_df["location_name"] == "basra", "uri"] = "https://en.wikipedia.org/wiki/Basra_Governorate"
iraq_provinces_df.loc[iraq_provinces_df["location_name"] == "duhok", "uri"] = "https://en.wikipedia.org/wiki/Duhok_Governorate"
iraq_provinces_df.loc[iraq_provinces_df["location_name"] == "kerbala", "uri"] = "https://en.wikipedia.org/wiki/Karbala_Governorate"
iraq_provinces_df.loc[iraq_provinces_df["location_name"] == "ninewa", "uri"] = "https://en.wikipedia.org/wiki/Nineveh_Governorate"
iraq_provinces_df.loc[iraq_provinces_df["location_name"] == "qadissiya", "uri"] = "https://en.wikipedia.org/wiki/Al-Q%C4%81disiyyah_Governorate"
iraq_provinces_df.loc[iraq_provinces_df["location_name"] == "salah al-din", "uri"] = "https://en.wikipedia.org/wiki/Saladin_Governorate"
iraq_provinces_df.loc[iraq_provinces_df["location_name"] == "thi qar", "uri"] = "https://en.wikipedia.org/wiki/Dhi_Qar_Governorate"
iraq_provinces_df.loc[iraq_provinces_df["location_name"] == "wassit", "uri"] = "https://en.wikipedia.org/wiki/Wasit_Governorate"

In [ ]:
# Jordan
jordan_provinces_df = pd.DataFrame(data={"location_name": jordan_provinces, "uri" : [er.getLocationUri(country) for country in jordan_provinces],"location_name_ara":jordan_provinces_ara, "adm" : 1, "country":"Jordan"})

jordan_provinces_df.loc[jordan_provinces_df["location_name"] == "ajloun", "uri"] = "https://en.wikipedia.org/wiki/Ajloun_Governorate"
jordan_provinces_df.loc[jordan_provinces_df["location_name"] == "jerash", "uri"] = "https://en.wikipedia.org/wiki/Jerash_Governorate"
jordan_provinces_df.loc[jordan_provinces_df["location_name"] == "karak", "uri"] = "https://en.wikipedia.org/wiki/Karak_Governorate"

In [ ]:
# Lebanon
lebanon_provinces_df = pd.DataFrame(data={"location_name": lebanon_provinces_shp_names["ADM1_EN_cl"].values, "uri" : [er.getLocationUri(country) for country in lebanon_provinces_shp_names["ADM1_EN_cl"].values],"location_name_ara":lebanon_provinces_shp_names["admin1Na_1"].values, "adm" : 1, "country":"Lebanon"})

lebanon_provinces_df.loc[lebanon_provinces_df["location_name"] == "akkar", "uri"] = "https://en.wikipedia.org/wiki/Akkar_Governorate"
lebanon_provinces_df.loc[lebanon_provinces_df["location_name"] == "baalbek-hermel", "uri"] = "https://en.wikipedia.org/wiki/Baalbek-Hermel_Governorate"
lebanon_provinces_df.loc[lebanon_provinces_df["location_name"] == "beirut", "uri"] = "https://en.wikipedia.org/wiki/Beirut_Governorate"
lebanon_provinces_df.loc[lebanon_provinces_df["location_name"] == "bekaa", "uri"] = "https://en.wikipedia.org/wiki/Beqaa_Governorate"
lebanon_provinces_df.loc[lebanon_provinces_df["location_name"] == "nabatieh", "uri"] = "https://en.wikipedia.org/wiki/Nabatieh_Governorate"
lebanon_provinces_df.loc[lebanon_provinces_df["location_name"] == "north lebanon", "uri"] = "https://en.wikipedia.org/wiki/North_Governorate"
lebanon_provinces_df.loc[lebanon_provinces_df["location_name"] == "south lebanon", "uri"] = "https://en.wikipedia.org/wiki/South_Governorate"

In [ ]:
# Palestine
palestine_provinces_df = pd.DataFrame(data={"location_name": palestine_provinces, "uri" : [er.getLocationUri(country) for country in palestine_provinces],"location_name_ara":["الضفة الغربية", "غزة"], "adm" : 1, "country":"Palestine"})

palestine_provinces_df.loc[palestine_provinces_df["location_name"] == "gaza strip", "location_name"] = "gaza"
gaza_df = palestine_provinces_df.loc[palestine_provinces_df["location_name"] == "gaza", ]
gaza_df = pd.concat([gaza_df, gaza_df, gaza_df], ignore_index=True)

gaza_df.loc[gaza_df.index == 0, "uri"] = "https://en.wikipedia.org/wiki/Gaza"
gaza_df.loc[gaza_df.index == 1, "uri"] = "http://en.wikipedia.org/wiki/Gaza_Governorate"
gaza_df.loc[gaza_df.index == 2, "uri"] = "https://en.wikipedia.org/wiki/Gaza_City"

palestine_provinces_df = pd.concat([palestine_provinces_df, gaza_df]).reset_index(drop=True)  

In [ ]:
# Syria
syria_provinces_df = pd.DataFrame(data={"location_name": syria_provinces_shp_names["ADM1_EN_cl"].values, "uri" : [er.getLocationUri(country) for country in syria_provinces_shp_names["ADM1_EN_cl"].values],"location_name_ara":syria_provinces_shp_names["ADM1_AR"].values, "adm" : 1, "country":"Syria"})

syria_provinces_df.loc[syria_provinces_df["location_name"] == "hama", "uri"] = "https://en.wikipedia.org/wiki/Hama_Governorate"
syria_provinces_df.loc[syria_provinces_df["location_name"] == "hasakeh", "uri"] = "https://en.wikipedia.org/wiki/Al-Hasakah_Governorate"
syria_provinces_df.loc[syria_provinces_df["location_name"] == "idleb", "uri"] = "https://en.wikipedia.org/wiki/Idlib_Governorate"
syria_provinces_df.loc[syria_provinces_df["location_name"] == "lattakia", "uri"] = "https://en.wikipedia.org/wiki/Latakia_Governorate"
syria_provinces_df.loc[syria_provinces_df["location_name"] == "rural damascus", "uri"] = "https://en.wikipedia.org/wiki/Rif_Dimashq_Governorate"
syria_provinces_df.loc[syria_provinces_df["location_name"] == "sweida", "uri"] = "https://en.wikipedia.org/wiki/As-Suwayda_Governorate"
syria_provinces_df.loc[syria_provinces_df["location_name"] == "tartous", "uri"] = "https://en.wikipedia.org/wiki/Tartus_Governorate"

In [ ]:
mashreq_countries_df = pd.concat([mashreq_countries_df, iraq_provinces_df, lebanon_provinces_df, syria_provinces_df, palestine_provinces_df, jordan_provinces_df]).reset_index(drop=True)

In [ ]:
mashreq_countries_df.to_csv(data_path_newsapi + "mashreq_countries_adm0_adm1.csv", index = False)

Adding admin level 2

In [ ]:
# Iraq
iraq_district_df = pd.DataFrame(data={"location_name": iraq_districts_shp["ADM2_EN_cl"].values, "uri" : [er.getLocationUri(country) for country in iraq_districts_shp["ADM2_EN_cl"].values], "location_name_ara" : iraq_districts_shp["ADM2_AR"].values, "adm" : 2, "country":"Iraq"})

iraq_district_df.loc[iraq_district_df["location_name"] == 'abu al-khaseeb', "uri"] = 'http://en.wikipedia.org/wiki/Abu_Al-Khaseeb_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'afaq', "uri"] = 'http://en.wikipedia.org/wiki/Afaq_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'ain al-tamur', "uri"] = 'http://en.wikipedia.org/wiki/Ain_Al-Tamur_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'adhamiya', "uri"] = 'http://en.wikipedia.org/wiki/Adhamiyah'
iraq_district_df.loc[iraq_district_df["location_name"] == 'amadiya', "uri"] = 'https://en.wikipedia.org/wiki/Amedi_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'amara', "uri"] = 'https://en.wikipedia.org/wiki/Amara_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'baaj', "uri"] = 'https://en.wikipedia.org/wiki/Al-Ba%27aj_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'basrah', "uri"] = 'http://en.wikipedia.org/wiki/Basrah_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'chibayish', "uri"] = 'http://en.wikipedia.org/wiki/Al-Chibayish_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'daur', "uri"] =  'https://en.wikipedia.org/wiki/Al-Daur_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'diwaniya', "uri"] = 'https://en.wikipedia.org/wiki/Diwaniya_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'falluja', "uri"] = 'https://en.wikipedia.org/wiki/Fallujah_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'faw', "uri"] = 'https://en.wikipedia.org/wiki/Al-Faw_District,_Basra_Governorate'
iraq_district_df.loc[iraq_district_df["location_name"] == 'hai', "uri"] = 'https://en.wikipedia.org/wiki/Al-Hai_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'hamdaniya', "uri"] = 'http://en.wikipedia.org/wiki/Al-Hamdaniya_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'hamza', "uri"] = 'http://en.wikipedia.org/wiki/Hamza_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'hashimiya', "uri"] = 'http://en.wikipedia.org/wiki/Hashimiya_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'hatra', "uri"] = 'https://en.wikipedia.org/wiki/Hatra_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'hawiga', "uri"] = 'http://en.wikipedia.org/wiki/Al-Hawiga_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'hilla', "uri"] = 'https://en.wikipedia.org/wiki/Al-Hilla_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'hindiya', "uri"] = 'https://en.wikipedia.org/wiki/Al-Hindiya_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'kadhmiyah', "uri"] = 'https://en.wikipedia.org/wiki/Kadhimiya'
iraq_district_df.loc[iraq_district_df["location_name"] == 'kahla', "uri"] = 'https://en.wikipedia.org/wiki/Al-Kahla_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'kaim', "uri"] = ''
iraq_district_df.loc[iraq_district_df["location_name"] == 'karkh', "uri"] = 'https://en.wikipedia.org/wiki/Karkh'
iraq_district_df.loc[iraq_district_df["location_name"] == 'khalis', "uri"] = 'https://en.wikipedia.org/wiki/Al_Khalis_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'khidhir', "uri"] = 'https://en.wikipedia.org/wiki/Al-Khidhir_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'kufa', "uri"] = 'http://en.wikipedia.org/wiki/Kufa_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'kut', "uri"] = 'https://en.wikipedia.org/wiki/Kut_District'
iraq_district_df.loc[iraq_district_df["location_name"] == "mada'in", "uri"] = 'https://en.wikipedia.org/wiki/Al-Mada%27in_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'mahaweel', "uri"] = 'https://en.wikipedia.org/wiki/Al-Mahawil_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'mahmoudiya', "uri"] = 'https://en.wikipedia.org/wiki/Mahmudiya_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'maimouna', "uri"] = 'https://en.wikipedia.org/wiki/Al-Maimouna_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'manathera', "uri"] = 'https://en.wikipedia.org/wiki/Al-Manathera_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'mejar al-kabir', "uri"] = 'https://en.wikipedia.org/wiki/Al-Mejar_Al-Kabir_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'midaina', "uri"] = 'https://en.wikipedia.org/wiki/Al-Midaina_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'mosul', "uri"] = 'https://en.wikipedia.org/wiki/Mosul_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'muqdadiya', "uri"] = 'https://en.wikipedia.org/wiki/Al-Miqdadiya_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'mussyab', "uri"] = 'https://en.wikipedia.org/wiki/Al-Musayab_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'najaf', "uri"] = 'https://en.wikipedia.org/wiki/Najaf_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'namaniya', "uri"] = 'https://en.wikipedia.org/wiki/Al-Nu%27maniya_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'nasiriya', "uri"] = 'https://en.wikipedia.org/wiki/Nasiriyah_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'qurna', "uri"] = 'https://en.wikipedia.org/wiki/Al-Qurna_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'ramadi', "uri"] = 'https://en.wikipedia.org/wiki/Ramadi_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'rifai', "uri"] = 'https://en.wikipedia.org/wiki/Al-Rifa%27i_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'risafa', "uri"] = 'https://en.wikipedia.org/wiki/Al-Rusafa,_Iraq'
iraq_district_df.loc[iraq_district_df["location_name"] == 'rumaitha', "uri"] = 'http://en.wikipedia.org/wiki/Al-Rumaitha_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'rutba', "uri"] = 'https://en.wikipedia.org/wiki/Ar-Rutba_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'salman', "uri"] = 'https://en.wikipedia.org/wiki/Al-Salman_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'samawa', "uri"] = 'https://en.wikipedia.org/wiki/As-Samawah_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'shamiya', "uri"] = 'https://en.wikipedia.org/wiki/Al-Shamiya_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'shatra', "uri"] = 'https://en.wikipedia.org/wiki/Al-Shatrah_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'shikhan', "uri"] = 'https://en.wikipedia.org/wiki/Shekhan_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'shirqat', "uri"] = 'https://en.wikipedia.org/wiki/Al-Shirqat_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'sulaymaniyah', "uri"] = 'https://en.wikipedia.org/wiki/Sulaymaniyah_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'suwaira', "uri"] = 'https://en.wikipedia.org/wiki/Al-Suwaira_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'thawra', "uri"] = 'https://en.wikipedia.org/wiki/Sadr_City'
iraq_district_df.loc[iraq_district_df["location_name"] == 'zibar', "uri"] = ''
iraq_district_df.loc[iraq_district_df["location_name"] == 'zubair', "uri"] = 'https://en.wikipedia.org/wiki/Al-Zubair_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'ali al-gharbi', "uri"] = 'https://en.wikipedia.org/wiki/Ali_Al-Gharbi_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'ana', "uri"] = 'https://en.wikipedia.org/wiki/Anah_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'aqra', "uri"] = ''
iraq_district_df.loc[iraq_district_df["location_name"] == 'badra', "uri"] = 'https://en.wikipedia.org/wiki/Badra_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'balad', "uri"] = 'https://en.wikipedia.org/wiki/Balad_District,_Iraq'
iraq_district_df.loc[iraq_district_df["location_name"] == 'baladruz', "uri"] = 'https://en.wikipedia.org/wiki/Balad_Ruz_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'baquba', "uri"] = 'https://en.wikipedia.org/wiki/Ba%27quba_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'beygee', "uri"] = ''
iraq_district_df.loc[iraq_district_df["location_name"] == 'chamchamal', "uri"] = 'https://en.wikipedia.org/wiki/Chamchamal_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'daquq', "uri"] = 'https://en.wikipedia.org/wiki/Daquq_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'derbendikhan', "uri"] = 'https://en.wikipedia.org/wiki/Darbandikhan_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'dibis', "uri"] = 'https://en.wikipedia.org/wiki/Dibis_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'dokan', "uri"] = 'https://en.wikipedia.org/wiki/Dokan_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'duhok', "uri"] = ''
iraq_district_df.loc[iraq_district_df["location_name"] == 'erbil', "uri"] = 'https://en.wikipedia.org/wiki/Erbil_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'haditha', "uri"] = 'https://en.wikipedia.org/wiki/Haditha_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'halabcha', "uri"] = 'https://en.wikipedia.org/wiki/Halabja'
iraq_district_df.loc[iraq_district_df["location_name"] == 'heet', "uri"] = 'https://en.wikipedia.org/wiki/Hit_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'kalar', "uri"] = 'https://en.wikipedia.org/wiki/Kalar_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'kerbela', "uri"] = 'https://en.wikipedia.org/wiki/Kerbala_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'khanaqin', "uri"] = 'https://en.wikipedia.org/wiki/Khanaqin_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'kifri', "uri"] = 'https://en.wikipedia.org/wiki/Kifri_District,_Diyala_Governorate'
iraq_district_df.loc[iraq_district_df["location_name"] == 'kirkuk', "uri"] = 'https://en.wikipedia.org/wiki/Kirkuk_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'koysinjaq', "uri"] = 'https://en.wikipedia.org/wiki/Koy_Sinjaq_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'makhmour', "uri"] = 'https://en.wikipedia.org/wiki/Makhmur_district'
iraq_district_df.loc[iraq_district_df["location_name"] == 'panjwin', "uri"] = 'https://en.wikipedia.org/wiki/Penjwen_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'pshdar', "uri"] = 'https://en.wikipedia.org/wiki/Pshdar_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'qalat saleh', "uri"] = 'https://en.wikipedia.org/wiki/Qal%27at_Saleh_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'rania', "uri"] = 'https://en.wikipedia.org/wiki/Ranya_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'rawanduz', "uri"] = 'https://en.wikipedia.org/wiki/Soran_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'samarra', "uri"] = 'https://en.wikipedia.org/wiki/Samarra_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'shaqlawa', "uri"] = 'http://en.wikipedia.org/wiki/Shaqlawa'
iraq_district_df.loc[iraq_district_df["location_name"] == 'sharbazher', "uri"] = 'https://en.wikipedia.org/wiki/Sharbazher_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'shat al-arab', "uri"] = ''
iraq_district_df.loc[iraq_district_df["location_name"] == 'sinjar', "uri"] = 'https://en.wikipedia.org/wiki/Sinjar_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'sumail', "uri"] = 'https://en.wikipedia.org/wiki/Simele_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'suq al-shoyokh', "uri"] = 'https://en.wikipedia.org/wiki/Suq_al-Shuyukh_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'telafar', "uri"] = 'https://en.wikipedia.org/wiki/Tel_Afar_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'tikrit', "uri"] = 'https://en.wikipedia.org/wiki/Tikrit_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'tilkaef', "uri"] = ''
iraq_district_df.loc[iraq_district_df["location_name"] == 'tooz khurmato', "uri"] = 'https://en.wikipedia.org/wiki/Tooz_District'
iraq_district_df.loc[iraq_district_df["location_name"] == 'zakho', "uri"] = 'https://en.wikipedia.org/wiki/Zakho_District'

In [ ]:
# Jordan
jordan_district_df = pd.DataFrame(data={"location_name": jordan_districts_shp["ADM2_EN_cl"].values, "uri" : [er.getLocationUri(country) for country in jordan_districts_shp["ADM2_EN_cl"].values], "location_name_ara" : jordan_districts_shp["Native"].values, "adm" : 2, "country":"Jordan"})

jordan_district_df.loc[jordan_district_df["location_name"] == 'kufranjah', "uri"] = 'https://en.wikipedia.org/wiki/Kufranjah'
jordan_district_df.loc[jordan_district_df["location_name"] == 'ajlun qasabah', "uri"] = 'https://en.wikipedia.org/wiki/Qa%E1%B9%A3abah_%27Ajl%C5%ABn'
jordan_district_df.loc[jordan_district_df["location_name"] == 'quairah', "uri"] = 'https://en.wikipedia.org/wiki/Al-Q%C5%ABa%C4%ABrah'
jordan_district_df.loc[jordan_district_df["location_name"] == 'aqaba qasabah', "uri"] = 'https://en.wikipedia.org/wiki/Qa%E1%B9%A3abah_al-%27Aqabah'
jordan_district_df.loc[jordan_district_df["location_name"] == 'ain albasha', "uri"] = ''
jordan_district_df.loc[jordan_district_df["location_name"] == 'shoonah janoobiyah', "uri"] = ''
jordan_district_df.loc[jordan_district_df["location_name"] == 'dair alla', "uri"] = 'https://en.wikipedia.org/wiki/Deir_Alla'
jordan_district_df.loc[jordan_district_df["location_name"] == 'fuhais and mahes', "uri"] = 'https://en.wikipedia.org/wiki/Fuheis'
jordan_district_df.loc[jordan_district_df["location_name"] == 'salt qasabah', "uri"] = 'https://en.wikipedia.org/wiki/As-Salt'
jordan_district_df.loc[jordan_district_df["location_name"] == 'aghwar janoobiyah', "uri"] = 'https://en.wikipedia.org/wiki/Al-%C4%80ghw%C4%81r_al-Jan%C5%ABb%C4%AB'
jordan_district_df.loc[jordan_district_df["location_name"] == 'mazar janoobee', "uri"] = 'https://en.wikipedia.org/wiki/Al-Maz%C4%81r_al-Jan%C5%ABb%C4%AB'
jordan_district_df.loc[jordan_district_df["location_name"] == 'qasr', "uri"] = 'https://en.wikipedia.org/wiki/Al-Qa%E1%B9%A3r'
jordan_district_df.loc[jordan_district_df["location_name"] == 'qatraneh', "uri"] = 'https://en.wikipedia.org/wiki/Al-Qatraneh'
jordan_district_df.loc[jordan_district_df["location_name"] == 'ayy', "uri"] = 'https://en.wikipedia.org/wiki/%27Ayy'
jordan_district_df.loc[jordan_district_df["location_name"] == "faqo'e", "uri"] = 'https://en.wikipedia.org/wiki/Faq%C5%AB%27e'
jordan_district_df.loc[jordan_district_df["location_name"] == 'karak', "uri"] = 'https://en.wikipedia.org/wiki/Qa%E1%B9%A3abah_al-Karak'
jordan_district_df.loc[jordan_district_df["location_name"] == 'badiah shamaliyah', "uri"] = 'https://en.wikipedia.org/wiki/Badiah_Shamaliyah'
jordan_district_df.loc[jordan_district_df["location_name"] == 'badiah gharbiyah', "uri"] = 'https://en.wikipedia.org/wiki/Badiah_Gharbiyah'
jordan_district_df.loc[jordan_district_df["location_name"] == 'rwaished', "uri"] = 'https://en.wikipedia.org/wiki/Ruwaishid_District'
jordan_district_df.loc[jordan_district_df["location_name"] == 'mafraq qasabah', "uri"] = 'https://en.wikipedia.org/wiki/Mafraq_Qasabah'
jordan_district_df.loc[jordan_district_df["location_name"] == "jami'ah", "uri"] = 'http://en.wikipedia.org/wiki/Al_Jami`ah'
jordan_district_df.loc[jordan_district_df["location_name"] == 'jizah', "uri"] = 'https://en.wikipedia.org/wiki/Al-Jizah,_Jordan'
jordan_district_df.loc[jordan_district_df["location_name"] == 'muaqqar', "uri"] = 'https://en.wikipedia.org/wiki/Al-Muwaqqar'
jordan_district_df.loc[jordan_district_df["location_name"] == 'quaismeh', "uri"] = 'https://en.wikipedia.org/wiki/Al-Quesmah'
jordan_district_df.loc[jordan_district_df["location_name"] == 'marka', "uri"] = 'http://en.wikipedia.org/wiki/Markazi_Province'
jordan_district_df.loc[jordan_district_df["location_name"] == "na'oor", "uri"] = 'https://en.wikipedia.org/wiki/Na%27our'
jordan_district_df.loc[jordan_district_df["location_name"] == 'amman qasabah', "uri"] = 'https://en.wikipedia.org/wiki/Amman'
jordan_district_df.loc[jordan_district_df["location_name"] == 'sahab', "uri"] = 'https://en.wikipedia.org/wiki/Sahab,_Jordan'
jordan_district_df.loc[jordan_district_df["location_name"] == 'wadi essier', "uri"] = 'https://en.wikipedia.org/wiki/Wadi_Al-Seer'
jordan_district_df.loc[jordan_district_df["location_name"] == 'hasa', "uri"] = 'https://en.wikipedia.org/wiki/Al-Hasa_District'
jordan_district_df.loc[jordan_district_df["location_name"] == 'bsaira', "uri"] = 'https://en.wikipedia.org/wiki/B%E1%B9%A3a%C4%ABr%C4%81'
jordan_district_df.loc[jordan_district_df["location_name"] == 'tafiela', "uri"] = 'https://en.wikipedia.org/wiki/Qa%E1%B9%A3abah_a%E1%B9%AD-%E1%B9%ACaf%C4%ABlah'
jordan_district_df.loc[jordan_district_df["location_name"] == 'hashemiyah', "uri"] = 'https://en.wikipedia.org/wiki/Al-H%C4%81shimiyah_District'
jordan_district_df.loc[jordan_district_df["location_name"] == 'russeifa', "uri"] = 'https://en.wikipedia.org/wiki/Russeifa'
jordan_district_df.loc[jordan_district_df["location_name"] == 'zarqa qasabah', "uri"] = 'https://en.wikipedia.org/wiki/Zarqa'
jordan_district_df.loc[jordan_district_df["location_name"] == 'aghwar shamaliyah', "uri"] = 'https://en.wikipedia.org/wiki/Al-%C4%80ghw%C4%81r_ash-Sham%C4%81liyah'
jordan_district_df.loc[jordan_district_df["location_name"] == 'koorah', "uri"] = 'https://en.wikipedia.org/wiki/Al-Kourah_District'
jordan_district_df.loc[jordan_district_df["location_name"] == 'mazar shamali', "uri"] = 'https://en.wikipedia.org/wiki/Al-Maz%C4%81r_ash-Sham%C4%81l%C4%AB'
jordan_district_df.loc[jordan_district_df["location_name"] == 'wastiyyah', "uri"] = 'https://en.wikipedia.org/wiki/Al-Was%E1%B9%AD%C4%AByah'
jordan_district_df.loc[jordan_district_df["location_name"] == 'ramtha', "uri"] = 'https://en.wikipedia.org/wiki/Ar-Ramtha_District'
jordan_district_df.loc[jordan_district_df["location_name"] == 'taybeh', "uri"] = 'https://en.wikipedia.org/wiki/A%E1%B9%AD-%E1%B9%ACa%C4%ABbah'
jordan_district_df.loc[jordan_district_df["location_name"] == 'bani kenanah', "uri"] = 'https://en.wikipedia.org/wiki/Bani_Kinanah_district'
jordan_district_df.loc[jordan_district_df["location_name"] == 'bani obeid', "uri"] = 'https://en.wikipedia.org/wiki/Ban%C4%AB_%27Obe%C4%ABd'
jordan_district_df.loc[jordan_district_df["location_name"] == 'irbid qasabah', "uri"] = 'https://en.wikipedia.org/wiki/Qa%E1%B9%A3abah_Irbid'
jordan_district_df.loc[jordan_district_df["location_name"] == 'jerash', "uri"] = 'https://en.wikipedia.org/wiki/Qa%E1%B9%A3abah_Jarash'
jordan_district_df.loc[jordan_district_df["location_name"] == 'petra', "uri"] = 'https://en.wikipedia.org/wiki/Al-Betr%C4%81%27'
jordan_district_df.loc[jordan_district_df["location_name"] == 'huseiniya', "uri"] = 'https://en.wikipedia.org/wiki/Al-%E1%B8%A4use%C4%ABniyah'
jordan_district_df.loc[jordan_district_df["location_name"] == 'shobak qasabah', "uri"] = 'https://en.wikipedia.org/wiki/Shoubak'
jordan_district_df.loc[jordan_district_df["location_name"] == "ma'an", "uri"] = "https://en.wikipedia.org/wiki/Qa%E1%B9%A3abah_Ma%27%C4%81n"
jordan_district_df.loc[jordan_district_df["location_name"] == 'dieban', "uri"] = 'https://en.wikipedia.org/wiki/Dhiban,_Jordan'
jordan_district_df.loc[jordan_district_df["location_name"] == 'madaba qasabah', "uri"] = 'https://en.wikipedia.org/wiki/Madaba'

In [ ]:
# Lebanon
lebanon_district_df = pd.DataFrame(data={"location_name": lebanon_districts_shp["ADM2_EN_cl"].values, "uri" : [er.getLocationUri(country) for country in lebanon_districts_shp["ADM2_EN_cl"].values], "location_name_ara" : lebanon_districts_shp["admin2Na_1"].values, "adm" : 2, "country":"Lebanon"})

lebanon_district_df.loc[lebanon_district_df["location_name"] == 'akkar', "uri"] = 'https://en.wikipedia.org/wiki/Akkar_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'hermel', "uri"] = 'https://en.wikipedia.org/wiki/Hermel_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'minieh-dennie', "uri"] = 'https://en.wikipedia.org/wiki/Miniyeh-Danniyeh_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'tripoli', "uri"] = 'https://en.wikipedia.org/wiki/Tripoli_District,_Lebanon'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'zgharta', "uri"] = 'https://en.wikipedia.org/wiki/Zgharta_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'baalbek', "uri"] = 'https://en.wikipedia.org/wiki/Baalbek_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'koura', "uri"] = 'https://en.wikipedia.org/wiki/Koura_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'batroun', "uri"] = 'https://en.wikipedia.org/wiki/Batroun_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'bcharre', "uri"] = 'https://en.wikipedia.org/wiki/Bsharri_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'jbeil', "uri"] = 'https://en.wikipedia.org/wiki/Byblos_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'kesrwane', "uri"] = 'https://en.wikipedia.org/wiki/Keserwan_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'meten', "uri"] = 'https://en.wikipedia.org/wiki/Matn_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'zahle', "uri"] = 'https://en.wikipedia.org/wiki/Zahl%C3%A9_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'beirut', "uri"] = 'https://en.wikipedia.org/wiki/Beirut'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'baabda', "uri"] = 'http://en.wikipedia.org/wiki/Baabda'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'aley', "uri"] = 'https://en.wikipedia.org/wiki/Aley_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'west bekaa', "uri"] = 'https://en.wikipedia.org/wiki/Western_Beqaa_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'chouf', "uri"] = 'https://en.wikipedia.org/wiki/Chouf_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'rachaya', "uri"] = 'https://en.wikipedia.org/wiki/Rashaya_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'jezzine', "uri"] = 'https://en.wikipedia.org/wiki/Jezzine_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'saida', "uri"] = ''
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'nabatieh', "uri"] = 'https://en.wikipedia.org/wiki/Nabatiyeh_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'hasbaya', "uri"] = 'https://en.wikipedia.org/wiki/Hasbaya_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'marjaayoun', "uri"] = 'https://en.wikipedia.org/wiki/Marjeyoun_District'
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'sour', "uri"] = ''
lebanon_district_df.loc[lebanon_district_df["location_name"] == 'bent jbeil', "uri"] = 'https://en.wikipedia.org/wiki/Bint_Jbeil_District'

In [ ]:
# Palestine
palestine_district_df = pd.DataFrame(data={"location_name": palestine_districts_shp["ADM2_EN_cl"].values, "uri" : [er.getLocationUri(country) for country in palestine_districts_shp["ADM2_EN_cl"].values], "location_name_ara" : palestine_districts_shp["Native"].values, "adm" : 2, "country":"Palestine"})

palestine_district_df.loc[palestine_district_df["location_name"] == 'jenin', "uri"] = 'https://en.wikipedia.org/wiki/Jenin_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'tubas', "uri"] = 'https://en.wikipedia.org/wiki/Tubas_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'tulkarm', "uri"] = 'https://en.wikipedia.org/wiki/Tulkarm_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'nablus', "uri"] = 'https://en.wikipedia.org/wiki/Nablus_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'qalqilya', "uri"] = 'https://en.wikipedia.org/wiki/Qalqilya_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'salfit', "uri"] = 'https://en.wikipedia.org/wiki/Salfit_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'ramallah', "uri"] = 'https://en.wikipedia.org/wiki/Ramallah_and_al-Bireh_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'jericho', "uri"] = 'https://en.wikipedia.org/wiki/Jericho_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'jerusalem', "uri"] = 'https://en.wikipedia.org/wiki/Jerusalem_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'bethlehem', "uri"] = 'https://en.wikipedia.org/wiki/Bethlehem_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'hebron', "uri"] = 'https://en.wikipedia.org/wiki/Hebron_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'north gaza', "uri"] ='https://en.wikipedia.org/wiki/North_Gaza_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'gaza', "uri"] = 'https://en.wikipedia.org/wiki/Gaza_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'deir al-balah', "uri"] = 'https://en.wikipedia.org/wiki/Deir_al-Balah_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'khan younis', "uri"] = 'https://en.wikipedia.org/wiki/Khan_Yunis_Governorate'
palestine_district_df.loc[palestine_district_df["location_name"] == 'rafah', "uri"] = 'https://en.wikipedia.org/wiki/Rafah_Governorate'

In [ ]:
# Syria
syria_district_df = pd.DataFrame(data={"location_name": syria_districts_shp["ADM2_EN_cl"].values, "uri" : [er.getLocationUri(country) for country in syria_districts_shp["ADM2_EN_cl"].values], "location_name_ara" : syria_districts_shp["NAME_AR"].values, "adm" : 2, "country":"Syria"})

syria_district_df.loc[syria_district_df["location_name"] == 'damascus', "uri"] = 'https://en.wikipedia.org/wiki/Damascus'
syria_district_df.loc[syria_district_df["location_name"] == 'jebel saman', "uri"] = ""
syria_district_df.loc[syria_district_df["location_name"] == 'bab', "uri"] =  'https://en.wikipedia.org/wiki/Al-Bab_District'
syria_district_df.loc[syria_district_df["location_name"] == 'afrin', "uri"] =  'https://en.wikipedia.org/wiki/Afrin_District'
syria_district_df.loc[syria_district_df["location_name"] == "a'zaz", "uri"] =  "https://en.wikipedia.org/wiki/Azaz_District"
syria_district_df.loc[syria_district_df["location_name"] == 'menbij', "uri"] = "https://en.wikipedia.org/wiki/Manbij_District"
syria_district_df.loc[syria_district_df["location_name"] == 'ain al arab', "uri"] = "https://en.wikipedia.org/wiki/Ayn_al-Arab_District"
syria_district_df.loc[syria_district_df["location_name"] == 'safira', "uri"] =  'https://en.wikipedia.org/wiki/As-Safira_District'
syria_district_df.loc[syria_district_df["location_name"] == 'jarablus', "uri"] =  "https://en.wikipedia.org/wiki/Jarabulus_District"
syria_district_df.loc[syria_district_df["location_name"] == 'rural damascus', "uri"] =  "https://en.wikipedia.org/wiki/Markaz_Rif_Dimashq_District"
syria_district_df.loc[syria_district_df["location_name"] == 'duma', "uri"] =  'https://en.wikipedia.org/wiki/Douma_District'
syria_district_df.loc[syria_district_df["location_name"] == 'qutayfah', "uri"] =  "https://en.wikipedia.org/wiki/Al-Qutayfah_District"
syria_district_df.loc[syria_district_df["location_name"] == 'at tall', "uri"] =  "https://en.wikipedia.org/wiki/Al-Tall_District"
syria_district_df.loc[syria_district_df["location_name"] == 'yabroud', "uri"] =  "https://en.wikipedia.org/wiki/Yabroud_District"
syria_district_df.loc[syria_district_df["location_name"] == 'an nabk', "uri"] =  "https://en.wikipedia.org/wiki/An-Nabek_District"
syria_district_df.loc[syria_district_df["location_name"] == 'zabdani', "uri"] =  "https://en.wikipedia.org/wiki/Al-Zabadani_District"
syria_district_df.loc[syria_district_df["location_name"] == 'qatana', "uri"] =  'https://en.wikipedia.org/wiki/Qatana_District'
syria_district_df.loc[syria_district_df["location_name"] == 'darayya', "uri"] =  'https://en.wikipedia.org/wiki/Darayya_District'
syria_district_df.loc[syria_district_df["location_name"] == 'homs', "uri"] =  'https://en.wikipedia.org/wiki/Homs_District'
syria_district_df.loc[syria_district_df["location_name"] == 'qusayr', "uri"] =  'https://en.wikipedia.org/wiki/Al-Qusayr_District'
syria_district_df.loc[syria_district_df["location_name"] == 'tall kalakh', "uri"] =  "https://en.wikipedia.org/wiki/Talkalakh_District"
syria_district_df.loc[syria_district_df["location_name"] == 'rastan', "uri"] =  'https://en.wikipedia.org/wiki/Al-Rastan_District'
syria_district_df.loc[syria_district_df["location_name"] == 'tadmor', "uri"] = 'https://en.wikipedia.org/wiki/Tadmur_District'
syria_district_df.loc[syria_district_df["location_name"] == 'makhrim', "uri"] =  ""
syria_district_df.loc[syria_district_df["location_name"] == 'hama', "uri"] =  'https://en.wikipedia.org/wiki/Hama_District'
syria_district_df.loc[syria_district_df["location_name"] == 'suqaylabiyah', "uri"] = 'https://en.wikipedia.org/wiki/Al-Suqaylabiyah_District'
syria_district_df.loc[syria_district_df["location_name"] == 'salamiyeh', "uri"] =  "https://en.wikipedia.org/wiki/Salamiyah_District"
syria_district_df.loc[syria_district_df["location_name"] == 'masyaf', "uri"] =  'https://en.wikipedia.org/wiki/Masyaf_District'
syria_district_df.loc[syria_district_df["location_name"] == 'muhradah', "uri"] =  "https://en.wikipedia.org/wiki/Mahardah_District"
syria_district_df.loc[syria_district_df["location_name"] == 'lattakia', "uri"] =  "https://en.wikipedia.org/wiki/Latakia_District"
syria_district_df.loc[syria_district_df["location_name"] == 'jablah', "uri"] =  "https://en.wikipedia.org/wiki/Jableh_District"
syria_district_df.loc[syria_district_df["location_name"] == 'haffa', "uri"] =  'https://en.wikipedia.org/wiki/Al-Haffah_District'
syria_district_df.loc[syria_district_df["location_name"] == 'qardaha', "uri"] =  'https://en.wikipedia.org/wiki/Qardaha_District'
syria_district_df.loc[syria_district_df["location_name"] == 'idleb', "uri"] =  "https://en.wikipedia.org/wiki/Idlib_District"
syria_district_df.loc[syria_district_df["location_name"] == "ma'ra", "uri"] =  'https://en.wikipedia.org/wiki/Ma%27arrat_al-Nu%27man_District'
syria_district_df.loc[syria_district_df["location_name"] == 'harim', "uri"] =  'https://en.wikipedia.org/wiki/Harem_District'
syria_district_df.loc[syria_district_df["location_name"] == 'jisr-ash-shugur', "uri"] =  "https://en.wikipedia.org/wiki/Jisr_al-Shughur_District"
syria_district_df.loc[syria_district_df["location_name"] == 'ariha', "uri"] =  'https://en.wikipedia.org/wiki/Ariha_District'
syria_district_df.loc[syria_district_df["location_name"] == 'hasakeh', "uri"] =  "https://en.wikipedia.org/wiki/Al-Hasakah_District"
syria_district_df.loc[syria_district_df["location_name"] == 'quamishli', "uri"] =  "https://en.wikipedia.org/wiki/Qamishli_District"
syria_district_df.loc[syria_district_df["location_name"] == 'malikeyyeh', "uri"] =  "https://en.wikipedia.org/wiki/Al-Malikiyah_District"
syria_district_df.loc[syria_district_df["location_name"] == 'ras al ain', "uri"] =  ""
syria_district_df.loc[syria_district_df["location_name"] == 'deir-ez-zor', "uri"] = 'https://en.wikipedia.org/wiki/Deir_ez-Zor_District'
syria_district_df.loc[syria_district_df["location_name"] == 'abu kamal', "uri"] =  'https://en.wikipedia.org/wiki/Abu_Kamal_District'
syria_district_df.loc[syria_district_df["location_name"] == 'mayadin', "uri"] =  'https://en.wikipedia.org/wiki/Mayadin_District'
syria_district_df.loc[syria_district_df["location_name"] == 'tartous', "uri"] =  "https://en.wikipedia.org/wiki/Tartus_District"
syria_district_df.loc[syria_district_df["location_name"] == 'qadmous', "uri"] =  ""
syria_district_df.loc[syria_district_df["location_name"] == 'safita', "uri"] =  'https://en.wikipedia.org/wiki/Safita_District'
syria_district_df.loc[syria_district_df["location_name"] == 'dreikish', "uri"] =  "https://en.wikipedia.org/wiki/Duraykish_District"
syria_district_df.loc[syria_district_df["location_name"] == 'sheikh badr', "uri"] =  "https://en.wikipedia.org/wiki/Al-Shaykh_Badr_District"
syria_district_df.loc[syria_district_df["location_name"] == 'raqqa', "uri"] =  'https://en.wikipedia.org/wiki/Raqqa_District'
syria_district_df.loc[syria_district_df["location_name"] == 'tell abiad', "uri"] =  "https://en.wikipedia.org/wiki/Tell_Abyad_District"
syria_district_df.loc[syria_district_df["location_name"] == 'thawrah', "uri"] =  "https://en.wikipedia.org/wiki/Al-Thawrah_District"
syria_district_df.loc[syria_district_df["location_name"] == "dar'a", "uri"] =  'https://en.wikipedia.org/wiki/Daraa_District'
syria_district_df.loc[syria_district_df["location_name"] == 'sanamayn', "uri"] =  "https://en.wikipedia.org/wiki/Al-Sanamayn_District"
syria_district_df.loc[syria_district_df["location_name"] == "izra'", "uri"] =  'https://en.wikipedia.org/wiki/Izra_District'
syria_district_df.loc[syria_district_df["location_name"] == 'sweida', "uri"] =  "https://en.wikipedia.org/wiki/As-Suwayda_District"
syria_district_df.loc[syria_district_df["location_name"] == 'salkhad', "uri"] =  'https://en.wikipedia.org/wiki/Salkhad_District'
syria_district_df.loc[syria_district_df["location_name"] == 'shahba', "uri"] =  'https://en.wikipedia.org/wiki/Shahba_District'
syria_district_df.loc[syria_district_df["location_name"] == 'quneitra', "uri"] =  'https://en.wikipedia.org/wiki/Quneitra_District'
syria_district_df.loc[syria_district_df["location_name"] == 'fiq', "uri"] =  'https://en.wikipedia.org/wiki/Fiq_District'
syria_district_df.loc[syria_district_df["location_name"] == 'banyas', "uri"] =  "https://en.wikipedia.org/wiki/Baniyas_District"

In [ ]:
mashreq_df_adm0_amd1 = pd.read_csv(data_path_newsapi + "mashreq_countries_adm0_adm1.csv")

In [ ]:
mashreq_df_adm0_amd1_amd2 = pd.concat([mashreq_df_adm0_amd1, iraq_district_df, jordan_district_df, lebanon_district_df, palestine_district_df, syria_district_df])
mashreq_df_adm0_amd1_amd2.reset_index(drop=True, inplace=True)

In [ ]:
mashreq_df_adm0_amd1_amd2.to_csv(data_path_newsapi + "mashreq_countries_adm0_adm1_adm2.csv", index=False)

Create query table for newsapi

In [ ]:
mashreq_df_adm0_amd1_amd2.loc[mashreq_df_adm0_amd1_amd2["adm"] == 2,"location_name"] = ""
mashreq_df_adm0_amd1_amd2.loc[mashreq_df_adm0_amd1_amd2["adm"] == 2,"location_name_ara"] = ""

mashreq_df_adm0_amd1_amd2 = mashreq_df_adm0_amd1_amd2.loc[~((mashreq_df_adm0_amd1_amd2["uri"] == "") & (mashreq_df_adm0_amd1_amd2["adm"] == 2))].reset_index(drop=True)

In [ ]:
mashreq_df_adm0_amd1_amd2.to_csv(data_path_newsapi + "mashreq_countries_query_file.csv", index=False)